In [1]:
pip install pandas

  Using cached pandas-3.0.5-cp313-cp313-win_amd64.whl.metadata (19 kB)
Using cached pandas-3.0.5-cp313-cp313-win_amd64.whl (9.8 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# Project root
ROOT = Path.cwd().parent

TRANSACTION_PATH = ROOT / "data" / "raw" / "train_transaction.csv"
IDENTITY_PATH = ROOT / "data" / "raw" / "train_identity.csv"

print("Transaction file:", TRANSACTION_PATH)
print("Identity file:", IDENTITY_PATH)

print("Transaction exists:", TRANSACTION_PATH.exists())
print("Identity exists:", IDENTITY_PATH.exists())

Transaction file: c:\Users\Gaurav Sehgal\OneDrive\Desktop\financial_fraud_detection_agent\data\raw\train_transaction.csv
Identity file: c:\Users\Gaurav Sehgal\OneDrive\Desktop\financial_fraud_detection_agent\data\raw\train_identity.csv
Transaction exists: True
Identity exists: True


In [3]:
transactions = pd.read_csv(TRANSACTION_PATH)
identity = pd.read_csv(IDENTITY_PATH)

print("Transaction shape:", transactions.shape)
print("Identity shape:", identity.shape)

Transaction shape: (590540, 394)
Identity shape: (144233, 41)


In [4]:
transactions["isFraud"].value_counts()

isFraud
0    569877
1     20663
Name: count, dtype: int64

In [5]:
fraud_rate = transactions["isFraud"].mean() * 100

print(f"Fraud rate: {fraud_rate:.2f}%")

Fraud rate: 3.50%


In [6]:
missing = (
    transactions.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

missing.head(30)

dist2    93.628374
D7       93.409930
D13      89.509263
D14      89.469469
D12      89.041047
D6       87.606767
D8       87.312290
D9       87.312290
V162     86.123717
V142     86.123717
V146     86.123717
V147     86.123717
V141     86.123717
V138     86.123717
V163     86.123717
V161     86.123717
V154     86.123717
V153     86.123717
V158     86.123717
V157     86.123717
V139     86.123717
V148     86.123717
V149     86.123717
V140     86.123717
V156     86.123717
V155     86.123717
V151     86.122701
V159     86.122701
V165     86.122701
V164     86.122701
dtype: float64

In [7]:
print(
    "Transaction IDs in transaction table:",
    transactions["TransactionID"].nunique()
)

print(
    "Transaction IDs in identity table:",
    identity["TransactionID"].nunique()
)

print(
    "Identity IDs also present in transactions:",
    identity["TransactionID"].isin(
        transactions["TransactionID"]
    ).mean() * 100
)

Transaction IDs in transaction table: 590540
Transaction IDs in identity table: 144233
Identity IDs also present in transactions: 100.0


In [8]:
merged = transactions.merge(
    identity,
    on="TransactionID",
    how="left"
)

print("Merged shape:", merged.shape)

Merged shape: (590540, 434)


In [9]:
transactions.groupby("isFraud")["TransactionAmt"].describe()

transactions.groupby("isFraud")["TransactionAmt"].mean()

transactions.groupby("isFraud")["TransactionAmt"].median()

isFraud
0    68.5
1    75.0
Name: TransactionAmt, dtype: float64

In [10]:
transactions.groupby("isFraud")["TransactionDT"].describe()


transactions.sort_values("TransactionDT")[
    ["TransactionID", "TransactionDT", "isFraud"]
].head()



transactions.sort_values("TransactionDT")[
    ["TransactionID", "TransactionDT", "isFraud"]
].tail()

,TransactionID,TransactionDT,isFraud
590535,3577535,15811047,0
590536,3577536,15811049,0
590537,3577537,15811079,0
590538,3577538,15811088,0
590539,3577539,15811131,0


In [11]:
print("TransactionDT min:", transactions["TransactionDT"].min())
print("TransactionDT max:", transactions["TransactionDT"].max())


print(
    transactions.groupby("isFraud")["TransactionDT"]
    .agg(["min", "max", "mean", "median"])
)


print(
    transactions.groupby("isFraud")["TransactionDT"]
    .agg(["min", "max", "mean", "median"])
)

TransactionDT min: 86400
TransactionDT max: 15811131
           min       max          mean     median
isFraud                                          
0        86400  15811131  7.360791e+06  7271678.0
1        89760  15810876  7.690033e+06  7575230.0
           min       max          mean     median
isFraud                                          
0        86400  15811131  7.360791e+06  7271678.0
1        89760  15810876  7.690033e+06  7575230.0


In [12]:
merged_check = transactions[["TransactionID", "isFraud"]].merge(
    identity[["TransactionID"]],
    on="TransactionID",
    how="left",
    indicator=True
)

merged_check["has_identity"] = (
    merged_check["_merge"] == "both"
).astype(int)

print(
    merged_check.groupby("isFraud")["has_identity"]
    .mean()
)

isFraud
0    0.233235
1    0.547742
Name: has_identity, dtype: float64


In [14]:
import sys
from pathlib import Path

# Add the project root to the path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.features.feature_engineering import add_basic_features

features_df = add_basic_features(
    transactions,
    identity
)

print(features_df.shape)

(590540, 401)


In [15]:
features_df[
    [
        "TransactionID",
        "TransactionAmt",
        "TransactionAmt_log",
        "transaction_hour",
        "transaction_day",
        "has_identity",
        "missing_card_info",
        "missing_address",
        "missing_value_count",
        "isFraud"
    ]
].head()

,TransactionID,TransactionAmt,TransactionAmt_log,transaction_hour,transaction_day,has_identity,missing_card_info,missing_address,missing_value_count,isFraud
0,2987000,68.5,4.241327,0,1,0,1,0,194,0
1,2987001,29.0,3.401197,0,1,0,0,0,190,0
2,2987002,59.0,4.094345,0,1,0,0,0,171,0
3,2987003,50.0,3.931826,0,1,0,0,0,187,0
4,2987004,50.0,3.931826,0,1,1,0,0,120,0


In [16]:
import sys
from pathlib import Path

# Add the project root to the path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))


from src.data.split import time_based_split

train_df, validation_df, test_df = time_based_split(
    features_df
)

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

Train: (413378, 401)
Validation: (88581, 401)
Test: (88581, 401)


In [17]:
print("TRAIN")
print(train_df["TransactionDT"].min())
print(train_df["TransactionDT"].max())

print("\nVALIDATION")
print(validation_df["TransactionDT"].min())
print(validation_df["TransactionDT"].max())

print("\nTEST")
print(test_df["TransactionDT"].min())
print(test_df["TransactionDT"].max())

TRAIN
86400
10437996

VALIDATION
10438003
13151840

TEST
13151880
15811131


In [18]:
for name, df in [
    ("Train", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:
    fraud_count = df["isFraud"].sum()
    fraud_rate = df["isFraud"].mean() * 100

    print(
        f"{name}: "
        f"{len(df):,} transactions | "
        f"{fraud_count:,} fraud | "
        f"{fraud_rate:.2f}% fraud"
    )

Train: 413,378 transactions | 14,538 fraud | 3.52% fraud
Validation: 88,581 transactions | 3,042 fraud | 3.43% fraud
Test: 88,581 transactions | 3,083 fraud | 3.48% fraud


In [ ]:
import sys
import importlib
from pathlib import Path

# Add the project root to the path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import and reload the module to get the latest version
import src.features.behavioural_features as bf_module
importlib.reload(bf_module)

from src.features.behavioural_features import (
    add_card_behavior_features,
    add_card_velocity_features,
    add_device_profile_features,
    
)

# Create behavioral features
behavior_df = train_df.copy()
behavior_df = add_card_behavior_features(behavior_df)
behavior_df = add_card_velocity_features(behavior_df)

print(behavior_df.shape)

(413378, 407)


In [36]:
behavior_df[
    [
        "TransactionID",
        "card1",
        "TransactionDT",
        "TransactionAmt",
        "card_transaction_count",
        "card_avg_amount",
        "amount_vs_card_avg",
        "new_card",
        "isFraud"
    ]
].head(20)

,TransactionID,card1,TransactionDT,TransactionAmt,card_transaction_count,card_avg_amount,amount_vs_card_avg,new_card,isFraud
0,3230924,1000,5787419,23.443,0,NaN,NaN,1,0
1,3023634,1001,916268,183.000,0,NaN,NaN,1,0
2,3151336,1001,3504180,29.000,1,183.000000,0.158470,0,0
3,3210739,1001,5270458,27.000,2,106.000000,0.254717,0,0
4,3020767,1004,842821,150.000,0,NaN,NaN,1,0
5,3028973,1004,1022173,30.000,1,150.000000,0.200000,0,0
6,3386444,1004,10082484,50.000,2,90.000000,0.555556,0,0
7,3038871,1005,1212802,50.000,0,NaN,NaN,1,0
8,3095681,1006,2145214,150.000,0,NaN,NaN,1,0
9,3234681,1006,5883179,150.000,1,150.000000,1.000000,0,0


In [37]:
print(
    behavior_df["card_txn_count_1h"].describe()
)

count    413378.000000
mean          1.358918
std           3.294610
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max          68.000000
Name: card_txn_count_1h, dtype: float64


In [38]:
print(
    behavior_df[
        ["card1", "TransactionDT", "card_txn_count_1h"]
    ].head(20)
)

    card1  TransactionDT  card_txn_count_1h
0    1000        5787419                  0
1    1001         916268                  0
2    1001        3504180                  0
3    1001        5270458                  0
4    1004         842821                  0
5    1004        1022173                  0
6    1004       10082484                  0
7    1005        1212802                  0
8    1006        2145214                  0
9    1006        5883179                  0
10   1007         850730                  0
11   1007        5651177                  0
12   1008        5845013                  0
13   1008        7241294                  0
14   1009         354228                  0
15   1009         926293                  0
16   1009        1222657                  0
17   1009        1539008                  0
18   1009        1786013                  0
19   1010        9826142                  0


In [39]:
print(behavior_df["card_txn_count_1h"].describe())

print(
    "Non-zero:",
    (behavior_df["card_txn_count_1h"] > 0).sum()
)

print(
    "Maximum:",
    behavior_df["card_txn_count_1h"].max()
)

count    413378.000000
mean          1.358918
std           3.294610
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max          68.000000
Name: card_txn_count_1h, dtype: float64
Non-zero: 176933
Maximum: 68


In [40]:
print(
    behavior_df[
        [
            "card_txn_count_1h",
            "card_txn_count_24h"
        ]
    ].describe()
)

       card_txn_count_1h  card_txn_count_24h
count      413378.000000       413378.000000
mean            1.358918           19.078761
std             3.294610           42.090811
min             0.000000            0.000000
25%             0.000000            1.000000
50%             0.000000            5.000000
75%             1.000000           20.000000
max            68.000000          654.000000


In [41]:
print(
    (
        behavior_df["card_txn_count_24h"]
        >= behavior_df["card_txn_count_1h"]
    ).all()
)

True


In [42]:
identity["DeviceInfo"].head()

0    SAMSUNG SM-G892A Build/NRD90M
1                       iOS Device
2                          Windows
3                              NaN
4                            MacOS
Name: DeviceInfo, dtype: str

In [ ]:
features_df[
    [
        "TransactionID",
        "TransactionAmt",
        "TransactionAmt_log",
        "transaction_hour",
        "transaction_day",
        "has_identity",
        "missing_card_info",
        "missing_address",
        "missing_value_count",
        "isFraud"
    ]
].head()

,TransactionID,TransactionAmt,TransactionAmt_log,transaction_hour,transaction_day,has_identity,missing_card_info,missing_address,missing_value_count,isFraud
0,2987000,68.5,4.241327,0,1,0,1,0,194,0
1,2987001,29.0,3.401197,0,1,0,0,0,190,0
2,2987002,59.0,4.094345,0,1,0,0,0,171,0
3,2987003,50.0,3.931826,0,1,0,0,0,187,0
4,2987004,50.0,3.931826,0,1,1,0,0,120,0


In [43]:
identity["DeviceInfo"].head()

0    SAMSUNG SM-G892A Build/NRD90M
1                       iOS Device
2                          Windows
3                              NaN
4                            MacOS
Name: DeviceInfo, dtype: str

In [44]:
print(
    "Unique devices:",
    identity["DeviceInfo"].nunique()
)

print(
    "Missing devices:",
    identity["DeviceInfo"].isna().mean() * 100
)

Unique devices: 1786
Missing devices: 17.726179168428864


In [45]:
device_data = identity[
    [
        "TransactionID",
        "DeviceInfo"
    ]
].copy()


behavior_df = behavior_df.merge(
    device_data,
    on="TransactionID",
    how="left"
)

In [46]:
behavior_df[
    [
        "TransactionID",
        "card1",
        "DeviceInfo"
    ]
].head(20)

,TransactionID,card1,DeviceInfo
0,3230924,1000,F3213 Build/36.0.A.2.146
1,3023634,1001,NaN
2,3151336,1001,NaN
3,3210739,1001,NaN
4,3020767,1004,MacOS
5,3028973,1004,Trident/7.0
6,3386444,1004,iOS Device
7,3038871,1005,Trident/7.0
8,3095681,1006,iOS Device
9,3234681,1006,rv:11.0


In [47]:
print(identity["DeviceInfo"].head())

print(
    "Unique devices:",
    identity["DeviceInfo"].nunique()
)

print(
    "Missing devices:",
    identity["DeviceInfo"].isna().mean() * 100
)

0    SAMSUNG SM-G892A Build/NRD90M
1                       iOS Device
2                          Windows
3                              NaN
4                            MacOS
Name: DeviceInfo, dtype: str
Unique devices: 1786
Missing devices: 17.726179168428864


In [55]:
device_data = identity[
    [
        "TransactionID",
        "DeviceInfo"
    ]
].copy()

In [56]:
behavior_df = behavior_df.drop(
    columns=["DeviceInfo"],
    errors="ignore"
)

behavior_df = behavior_df.merge(
    device_data,
    on="TransactionID",
    how="left"
)

In [57]:
behavior_df = add_device_profile_features(
    behavior_df
)

In [58]:
behavior_df[
    [
        "TransactionID",
        "card1",
        "DeviceInfo",
        "has_device_info",
        "device_profile_count",
        "device_profile_unique_cards",
        "new_device_profile",
        "isFraud"
    ]
].head(30)

,TransactionID,card1,DeviceInfo,has_device_info,device_profile_count,device_profile_unique_cards,new_device_profile,isFraud
0,3105398,14128,0PAJ5,1,0,0,1,0
1,3091537,11942,0PJA2,1,0,0,1,0
2,3087675,10027,0PM92,1,0,0,1,0
3,3094705,6174,0PM92,1,1,1,0,0
4,3306333,3702,0PM92,1,2,2,0,1
5,3306337,1976,0PM92,1,3,3,0,1
6,3142945,11195,1016S,1,0,0,1,0
7,3021070,3682,2PQ93,1,0,0,1,0
8,3023129,15497,2PS64 Build/NRD90M,1,0,0,1,0
9,3059407,14290,2PS64 Build/NRD90M,1,1,1,0,0


In [59]:
print(
    behavior_df[
        [
            "device_profile_count",
            "device_profile_unique_cards"
        ]
    ].describe()
)

       device_profile_count  device_profile_unique_cards
count         413378.000000                413378.000000
mean            2114.927763                   366.309013
std             6298.501821                   948.069941
min                0.000000                     0.000000
25%                0.000000                     0.000000
50%                0.000000                     0.000000
75%                0.000000                     0.000000
max            36667.000000                  4348.000000


In [60]:
print(
    behavior_df.groupby("isFraud")[
        [
            "has_device_info",
            "device_profile_count",
            "device_profile_unique_cards",
            "new_device_profile"
        ]
    ].mean()
)

         has_device_info  device_profile_count  device_profile_unique_cards  \
isFraud                                                                       
0               0.213522           2030.255471                   354.671104   
1               0.426537           4437.853694                   685.587013   

         new_device_profile  
isFraud                      
0                  0.003658  
1                  0.005984  


In [62]:
import sys
import importlib
from pathlib import Path

# Add the project root to the path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import and reload the module to get the latest version
import src.features.behavioural_features as bf_module
importlib.reload(bf_module)
from src.features.behavioural_features import add_card_device_features

In [63]:
behavior_df = add_card_device_features(
    behavior_df
)

In [64]:
behavior_df[
    [
        "TransactionID",
        "card1",
        "DeviceInfo",
        "card_device_transaction_count",
        "card_device_seen_before",
        "isFraud"
    ]
].head(30)

,TransactionID,card1,DeviceInfo,card_device_transaction_count,card_device_seen_before,isFraud
0,3105398,14128,0PAJ5,0,0,0
1,3091537,11942,0PJA2,0,0,0
2,3306337,1976,0PM92,0,0,1
3,3306333,3702,0PM92,0,0,1
4,3094705,6174,0PM92,0,0,0
5,3087675,10027,0PM92,0,0,0
6,3142945,11195,1016S,0,0,0
7,3021070,3682,2PQ93,0,0,0
8,3103301,4029,2PS64 Build/NRD90M,0,0,0
9,3064774,4577,2PS64 Build/NRD90M,0,0,0


In [65]:
print(
    behavior_df.groupby("isFraud")[
        [
            "card_device_transaction_count",
            "card_device_seen_before"
        ]
    ].mean()
)

         card_device_transaction_count  card_device_seen_before
isFraud                                                        
0                            17.727710                 0.162541
1                            32.159375                 0.329413


In [ ]:
import sys
import importlib
from pathlib import Path

# Add the project root to the path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
# Import and reload the module to get the latest version
import src.models.features as features_module
importlib.reload(features_module)

# Define INITIAL_FEATURES based on available columns
INITIAL_FEATURES = [col for col in behavior_df.columns if col not in ['TransactionID', 'isFraud']]

from src.models.features import INITIAL_FEATURES

X = behavior_df[INITIAL_FEATURES].copy()
y = behavior_df["isFraud"].copy()

ImportError: cannot import name 'INITIAL_FEATURES' from 'src.models.features' (c:\Users\Gaurav Sehgal\OneDrive\Desktop\financial_fraud_detection_agent\src\models\features.py)